In [3]:
import pandas as pd

# Load both files
ingredients = pd.read_csv('recipe_ingredients.csv')
recipes = pd.read_csv('recipes_master.csv')

# Aggregate ingredients with quantity
agg_ingredients = ingredients.groupby('recipe_id').agg(
    ingredients_list=('ingredient_name', lambda x: ', '.join(
        x + ' (' + ingredients.loc[x.index, 'quantity'] + ')'
    )),
    ingredient_count=('ingredient_name', 'count')
).reset_index()

# Merge with recipes
merged = pd.merge(recipes, agg_ingredients, on='recipe_id', how='inner')

# Save to CSV
merged.to_csv('recipes_final.csv', index=False)

In [4]:
merged.to_excel('recipes_final.xlsx', index=False)

In [5]:
merged_filtered = merged[['recipe_id', 'recipe_name', 'ingredients_list', 'ingredient_count', 'rating']]

merged_filtered.to_excel('recipes_final_filtered.xlsx', index=False)

In [23]:
import pandas as pd

# Load files
ingredients = pd.read_csv('recipe_ingredients.csv')
carbon = pd.read_excel('Wolfram_Food_Carbon_Footprint.xlsx')

# Clean wolfram ingredients
wolfram_ingredients = set(carbon['Clean Name'].dropna().str.lower().str.strip())

# Clean recipe ingredients
ingredients['ingredient_lower'] = ingredients['ingredient_name'].str.lower().str.strip()

# Manual map
manual_map = {
    'lentils (dal)': 'lentil',
    'mint leaves': 'fresh mint',
    'fresh coriander': 'coriander seed',
    'tamarind paste': 'tamarind',
    'sesame seeds': 'sesame',
    'coriander': 'coriander seed',
    'green chili': 'chili pepper',
    'bell pepper': 'pepper',
    'oil': 'sesame oil',
    'cooking oil': 'sesame oil',
    'salt': 'sea salt',
    'all-purpose flour': 'wheat flour',
    'red chili powder': 'chili pepper',
    'garam masala': 'spice',
    'black pepper': 'black peppercorn',
    'cloves': 'clove',
    'milk': 'milks',
    'water': 'bottled water',
}

ingredients['ingredient_mapped'] = ingredients['ingredient_lower'].replace(manual_map)
ingredients['in_wolfram'] = ingredients['ingredient_mapped'].isin(wolfram_ingredients)

# Match summary per recipe
match_summary = ingredients.groupby('recipe_id').agg(
    total_ingredients=('ingredient_name', 'count'),
    matched_ingredients=('in_wolfram', 'sum')
).reset_index()

match_summary['match_pct'] = (match_summary['matched_ingredients'] / match_summary['total_ingredients'] * 100).round(1)
match_summary['fully_matched'] = match_summary['matched_ingredients'] == match_summary['total_ingredients']

print(f"Total recipes: {len(match_summary)}")
print(f"Fully matched: {match_summary['fully_matched'].sum()}")
print(f"% fully matched: {match_summary['fully_matched'].mean()*100:.1f}%")
print(f"\nMatch % distribution:")
print(match_summary['match_pct'].describe())

Total recipes: 9997
Fully matched: 1297
% fully matched: 13.0%

Match % distribution:
count    9997.000000
mean       79.780824
std        12.384559
min        33.300000
25%        71.400000
50%        80.000000
75%        87.500000
max       100.000000
Name: match_pct, dtype: float64


In [ ]:
unmatched = ingredients[ingredients['in_wolfram'] == False]
top_unmatched = (unmatched.groupby('ingredient_mapped')
                 .size()
                 .reset_index(name='count')
                 .sort_values('count', ascending=False)
                 .head(20))
print(top_unmatched.to_string())

   ingredient_mapped  count
10            paneer   3100
5               eggs   2104
6   fenugreek leaves   2050
4       curry leaves   2004
3          chickpeas   1994
7               fish   1746
13          semolina   1597
8               ghee   1353
9      mustard seeds   1138
0            almonds   1116
2            cashews   1109
12           raisins   1069
1           bay leaf    814
11              peas    758


In [25]:
import pandas as pd

# Load files
ingredients = pd.read_csv('recipe_ingredients.csv')
carbon = pd.read_excel('Wolfram_Food_Carbon_Footprint.xlsx')

# Clean wolfram ingredients
wolfram_ingredients = set(carbon['Clean Name'].dropna().str.lower().str.strip())

# Clean recipe ingredients
ingredients['ingredient_lower'] = ingredients['ingredient_name'].str.lower().str.strip()

manual_map = {
    # previous mappings
    'lentils (dal)': 'lentil',
    'mint leaves': 'fresh mint',
    'fresh coriander': 'coriander seed',
    'tamarind paste': 'tamarind',
    'sesame seeds': 'sesame',
    'coriander': 'coriander seed',
    'green chili': 'chili pepper',
    'bell pepper': 'pepper',
    'oil': 'sesame oil',
    'cooking oil': 'sesame oil',
    'salt': 'sea salt',
    'all-purpose flour': 'wheat flour',
    'red chili powder': 'chili pepper',
    'garam masala': 'spice',
    'black pepper': 'black peppercorn',
    'cloves': 'clove',
    'milk': 'milks',
    'water': 'bottled water',
    # new mappings
    'eggs': 'egg',
    'fenugreek leaves': 'fenugreek seed',
    'curry leaves': 'curry powder',
    'chickpeas': 'chickpea',
    'fish': 'fishes',
    'ghee': 'butter',
    'mustard seeds': 'mustard',
    'almonds': 'almond',
    'cashews': 'cashew',
    'raisins': 'raisin',
    'bay leaf': 'herb',
    'peas': 'pea',
}

ingredients['ingredient_mapped'] = ingredients['ingredient_lower'].replace(manual_map)
ingredients['in_wolfram'] = ingredients['ingredient_mapped'].isin(wolfram_ingredients)

# Match summary per recipe
match_summary = ingredients.groupby('recipe_id').agg(
    total_ingredients=('ingredient_name', 'count'),
    matched_ingredients=('in_wolfram', 'sum')
).reset_index()

match_summary['match_pct'] = (match_summary['matched_ingredients'] / match_summary['total_ingredients'] * 100).round(1)
match_summary['fully_matched'] = match_summary['matched_ingredients'] == match_summary['total_ingredients']

print(f"Total recipes: {len(match_summary)}")
print(f"Fully matched: {match_summary['fully_matched'].sum()}")
print(f"% fully matched: {match_summary['fully_matched'].mean()*100:.1f}%")
print(f"\nMatch % distribution:")
print(match_summary['match_pct'].describe())

Total recipes: 9997
Fully matched: 6313
% fully matched: 63.1%

Match % distribution:
count    9997.000000
mean       95.650675
std         6.581849
min        57.100000
25%        91.700000
50%       100.000000
75%       100.000000
max       100.000000
Name: match_pct, dtype: float64


In [26]:
unmatched = ingredients[ingredients['in_wolfram'] == False]
top_unmatched = (unmatched.groupby('ingredient_mapped')
                 .size()
                 .reset_index(name='count')
                 .sort_values('count', ascending=False)
                 .head(20))
print(top_unmatched.to_string())

  ingredient_mapped  count
0            paneer   3100
1          semolina   1597


In [30]:
import pandas as pd
import requests
from io import StringIO

# =========================================================
# STEP 1: LOAD OWID DATASET
# =========================================================
def load_owid():
    url = "https://ourworldindata.org/grapher/ghg-per-kg-poore.csv?v=1&csvType=full"
    session = requests.Session()
    session.headers.update({"User-Agent": "Mozilla/5.0"})
    response = session.get(url, timeout=20)
    response.raise_for_status()
    df = pd.read_csv(StringIO(response.text))
    latest_year = df["Year"].max()
    df = df[df["Year"] == latest_year].copy()
    co2_col = [c for c in df.columns if c not in {"Entity", "Code", "Year"}][0]
    return df, co2_col

owid_df, co2_col = load_owid()

# Ingredients to fetch from OWID (not in Wolfram)
owid_map = {
    'paneer':   'Cheese',
    'semolina': 'Wheat & Rye',
    'bay leaf': 'Other Pulses',
}

# Build OWID CO2 lookup
owid_co2 = {}
for ingredient, entity in owid_map.items():
    row = owid_df[owid_df["Entity"].str.lower() == entity.lower()]
    if not row.empty:
        owid_co2[ingredient] = round(float(row.iloc[0][co2_col]), 2)

print("OWID values fetched:")
for k, v in owid_co2.items():
    print(f"  {k}: {v} kg CO2e/kg")

# =========================================================
# STEP 2: LOAD FILES
# =========================================================
ingredients = pd.read_csv('recipe_ingredients.csv')
carbon = pd.read_excel('Wolfram_Food_Carbon_Footprint.xlsx')

# =========================================================
# STEP 3: WOLFRAM MATCHING (only ingredients in Wolfram)
# =========================================================
wolfram_ingredients = set(carbon['Clean Name'].dropna().str.lower().str.strip())
ingredients['ingredient_lower'] = ingredients['ingredient_name'].str.lower().str.strip()

manual_map = {
    'lentils (dal)':    'lentil',
    'mint leaves':      'fresh mint',
    'fresh coriander':  'coriander seed',
    'tamarind paste':   'tamarind',
    'sesame seeds':     'sesame',
    'coriander':        'coriander seed',
    'green chili':      'chili pepper',
    'bell pepper':      'pepper',
    'oil':              'sesame oil',
    'cooking oil':      'sesame oil',
    'all-purpose flour':'wheat flour',
    'red chili powder': 'chili pepper',
    'garam masala':     'spice',
    'black pepper':     'black peppercorn',
    'cloves':           'clove',
    'milk':             'milks',
    'water':            'bottled water',
    'eggs':             'egg',
    'fenugreek leaves': 'fenugreek seed',
    'curry leaves':     'curry powder',
    'chickpeas':        'chickpea',
    'fish':             'fishes',
    'mustard seeds':    'mustard',
    'almonds':          'almond',
    'cashews':          'cashew',
    'raisins':          'raisin',
    'peas':             'pea',
    'ghee':             'butter',
    'salt':             'sea salt',
}

ingredients['ingredient_mapped'] = ingredients['ingredient_lower'].replace(manual_map)
ingredients['in_wolfram'] = ingredients['ingredient_mapped'].isin(wolfram_ingredients)

# =========================================================
# STEP 4: OWID FALLBACK
# =========================================================
ingredients['in_owid'] = (
    ~ingredients['in_wolfram'] &
    ingredients['ingredient_lower'].isin(owid_co2.keys())
)

ingredients['matched'] = ingredients['in_wolfram'] | ingredients['in_owid']

# =========================================================
# STEP 5: MATCH SUMMARY
# =========================================================
match_summary = ingredients.groupby('recipe_id').agg(
    total_ingredients=('ingredient_name', 'count'),
    matched_ingredients=('matched', 'sum')
).reset_index()

match_summary['match_pct'] = (match_summary['matched_ingredients'] / match_summary['total_ingredients'] * 100).round(1)
match_summary['fully_matched'] = match_summary['matched_ingredients'] == match_summary['total_ingredients']

print(f"\nTotal recipes: {len(match_summary)}")
print(f"Fully matched: {match_summary['fully_matched'].sum()}")
print(f"% fully matched: {match_summary['fully_matched'].mean()*100:.1f}%")
print(f"\nMatch % distribution:")
print(match_summary['match_pct'].describe())

OWID values fetched:
  paneer: 23.88 kg CO2e/kg
  semolina: 1.57 kg CO2e/kg
  bay leaf: 1.79 kg CO2e/kg

Total recipes: 9997
Fully matched: 9997
% fully matched: 100.0%

Match % distribution:
count    9997.0
mean      100.0
std         0.0
min       100.0
25%       100.0
50%       100.0
75%       100.0
max       100.0
Name: match_pct, dtype: float64


In [32]:
print(match_summary.head(10).to_string())

  recipe_id  total_ingredients  matched_ingredients  match_pct  fully_matched
0  RCP00001                 11                   11      100.0           True
1  RCP00002                  7                    7      100.0           True
2  RCP00003                  8                    8      100.0           True
3  RCP00004                  8                    8      100.0           True
4  RCP00005                 12                   12      100.0           True
5  RCP00006                 12                   12      100.0           True
6  RCP00007                  9                    9      100.0           True
7  RCP00008                  7                    7      100.0           True
8  RCP00009                  7                    7      100.0           True
9  RCP00010                  7                    7      100.0           True


In [33]:
recipes = pd.read_csv('recipes_master.csv')

result = ingredients.merge(recipes[['recipe_id', 'recipe_name']], on='recipe_id', how='left')

print(result[['recipe_id', 'recipe_name', 'ingredient_name', 'matched']].head(10).to_string())

  recipe_id recipe_name ingredient_name  matched
0  RCP00001       Sajji           Onion     True
1  RCP00001       Sajji          Garlic     True
2  RCP00001       Sajji            Salt     True
3  RCP00001       Sajji          Shrimp     True
4  RCP00001       Sajji         Chicken     True
5  RCP00001       Sajji             Oil     True
6  RCP00001       Sajji     Wheat Flour     True
7  RCP00001       Sajji           Sugar     True
8  RCP00001       Sajji   Mustard Seeds     True
9  RCP00001       Sajji          Mutton     True
